# 11 — Where was the scanner? Label-free scanner position for every station (FEAS-005)

The 2D module renders pinhole views **from the scanner position**. So far that position came from the empty disc under the tripod; when no disc is found (S164, S168, S179 and **all three test stations S170, S171, S180**) the centre of the floor was used. CODA-002 found no Switch and no correct Outlet in the test stations — views from a wrong place are one likely reason (objects hidden behind others, distorted, far away).

Two label-free estimates, checked against the tripod disc where it exists:

- **A — scan lines:** if the points are stored in scanner order, every short run of consecutive points lies in a vertical plane through the scanner; the planes' least-squares intersection gives (x, y). Only valid if the file keeps scanner order (small median step between consecutive points; the line fit rejects a shuffled file by itself).
- **B — point spacing:** a terrestrial scanner samples equal angles, so the nearest-neighbour spacing grows with the range from the real scanner; the (x, y) where log(spacing) − log(range) is most constant is the scanner (height fixed at 1.6 m above the floor).
- **diagnostic:** fraction of points visible from each estimate (from the real scanner every point is visible).

Synthetic test (room 10 × 8 m with a pillar, 0.6° scan): A error 0.07 mm, B error 9 cm; A refuses a shuffled file.

**Output:** a table per station (disc / floor centre / A / B, distances between them), `scanner_positions.json` (for notebook 10 version 4), and 360° intensity panoramas from the old and the new position (`pano_<station>.png`). CPU only — **Accelerator: None**, Internet **on**; about 20–40 min.

In [ ]:
import os, sys, json, time, subprocess
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy import ndimage
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CFG = dict(run_name="FEAS-005_scanner_position", gold=["S164", "S165", "S168", "S178", "S179", "S163", "S169"],
           test=["S170", "S171", "S180"], scanner_height_m=1.6, spacing_n=200000, vis_n=300000, seed=0,
           pano_deg_per_px=0.25)
OFFSET = np.array([900.0, 480.0, 100.0])
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.path.join("/kaggle/working" if ON_KAGGLE else "./results", CFG["run_name"])
RAW = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
os.makedirs(OUT, exist_ok=True); os.makedirs(RAW, exist_ok=True)
print(json.dumps(CFG, indent=1))

In [ ]:
# ---- helper functions (notebooks 06 / 09 and the new scanner-position estimates) ----
def instances(xyz, lab, c, link=0.03, min_pts=10):
    """Split the points of class c into separate objects: points closer than `link` metres are joined.
    Returns a list of index arrays (into xyz), largest first, objects with fewer than min_pts points dropped."""
    idx = np.flatnonzero(lab == c)
    if len(idx) == 0:
        return []
    p = xyz[idx]
    pairs = cKDTree(p).query_pairs(link, output_type="ndarray")
    n = len(idx)
    if len(pairs):
        g = coo_matrix((np.ones(len(pairs), dtype=np.int8), (pairs[:, 0], pairs[:, 1])), shape=(n, n))
        _, comp = connected_components(g, directed=False)
    else:
        comp = np.arange(n)
    out = [idx[comp == k] for k in np.unique(comp)]
    out = [o for o in out if len(o) >= min_pts]
    return sorted(out, key=len, reverse=True)


def describe(xyz, inst):
    """Size of one object: extent along its own principal axes (m), number of points, median point spacing (m)."""
    p = xyz[inst]
    q = p - p.mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    ext = np.sort(np.ptp(q @ vt.T, axis=0))[::-1]
    sp = float(np.median(cKDTree(p).query(p, k=2)[0][:, 1])) if len(p) > 1 else float("nan")
    return dict(points=int(len(p)), extent_m=[round(float(e), 3) for e in ext], spacing_m=round(sp, 4),
                centre=[round(float(v), 3) for v in p.mean(0)])


def view_frame(xyz, tree, inst, radius=0.30):
    """Camera looking straight at the surface the object sits on.
    n = normal of the surrounding surface (PCA of the neighbourhood without the object), pointing towards the
    side the object sticks out to; u, v = image axes in that surface (v as close to 'up' as possible)."""
    c = xyz[inst].mean(0)
    nb = np.asarray(tree.query_ball_point(c, radius))
    bg = np.setdiff1d(nb, inst, assume_unique=False)
    if len(bg) < 30:
        bg = nb
    q = xyz[bg] - xyz[bg].mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    n = vt[2]
    p0 = xyz[bg].mean(0)
    if (c - p0) @ n < 0:
        n = -n
    up = np.array([0.0, 0.0, 1.0])
    if abs(n @ up) > 0.9:                       # ceiling / floor: any horizontal axis will do
        up = np.array([0.0, 1.0, 0.0])
    u = np.cross(up, n); u /= np.linalg.norm(u)
    v = np.cross(n, u)
    return c, p0, n, u, v


def render_ortho(xyz, rgb, inten, lab, tree, centre, p0, n, u, v, half, px, depth=(-0.06, 0.20)):
    """Orthographic picture of everything within `half` metres of `centre` (in the image plane) and within the
    depth band in front of / behind the surface. For each pixel the point nearest to the viewer is kept.
    Returns rgb [H,W,3] uint8, intensity [H,W] float (nan = empty), label [H,W] int (-1 = empty)."""
    nb = np.asarray(tree.query_ball_point(centre, half * 1.5 + depth[1]))
    p = xyz[nb]
    d = (p - p0) @ n
    a = (p - centre) @ u
    b = (p - centre) @ v
    m = (d > depth[0]) & (d < depth[1]) & (np.abs(a) < half) & (np.abs(b) < half)
    nb, d, a, b = nb[m], d[m], a[m], b[m]
    W = int(np.ceil(2 * half / px))
    col = np.clip(((a + half) / px).astype(int), 0, W - 1)
    row = np.clip(((half - b) / px).astype(int), 0, W - 1)
    key = row * W + col
    order = np.lexsort((-d, key))               # per pixel: nearest to the viewer first
    ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    img = np.full((W * W, 3), 255, np.uint8); I = np.full(W * W, np.nan, np.float32); L = np.full(W * W, -1, np.int16)
    img[key[sel]] = rgb[nb[sel]]; I[key[sel]] = inten[nb[sel]]
    if lab is not None:
        L[key[sel]] = lab[nb[sel]]
    return img.reshape(W, W, 3), I.reshape(W, W), L.reshape(W, W)


def fill_small_holes(img, I, L, it=1):
    """Fill empty pixels from their neighbours (nearest filled pixel within `it` pixels) - display only."""
    empty = np.isnan(I)
    if not empty.any() or empty.all():
        return img, I, L
    dist, (ri, ci) = ndimage.distance_transform_edt(empty, return_indices=True)
    ok = empty & (dist <= it)
    img = img.copy(); I = I.copy(); L = L.copy()
    img[ok] = img[ri[ok], ci[ok]]; I[ok] = I[ri[ok], ci[ok]]; L[ok] = L[ri[ok], ci[ok]]
    return img, I, L


def estimate_scanner_xy(xyz, lab, floor_class=12, cell=0.02):
    """A terrestrial scanner leaves an unscanned disc on the floor under its tripod. Find the largest empty disc
    that is completely surrounded by floor points. Returns (x, y, radius_m, floor_z) or None."""
    f = xyz[lab == floor_class]
    if len(f) < 1000:
        return None
    lo = f[:, :2].min(0)
    g = ((f[:, :2] - lo) / cell).astype(int)
    occ = np.zeros(g.max(0) + 1, bool); occ[g[:, 0], g[:, 1]] = True
    occ = ndimage.binary_closing(occ, iterations=1)
    holes = ndimage.binary_fill_holes(occ) & ~occ
    if not holes.any():
        return None
    lab_h, nh = ndimage.label(holes)
    dist = ndimage.distance_transform_edt(holes)
    i, j = np.unravel_index(np.argmax(dist), dist.shape)
    r = float(dist[i, j] * cell)
    return float(lo[0] + (i + 0.5) * cell), float(lo[1] + (j + 0.5) * cell), r, float(np.median(f[:, 2]))


def render_panorama(xyz, rgb, inten, lab, origin, deg_per_px=0.1, max_range=30.0):
    """Equirectangular 360 x 180 degree picture seen from `origin`; per pixel the nearest point is kept."""
    q = xyz - origin
    r = np.linalg.norm(q, axis=1)
    m = (r > 0.2) & (r < max_range)
    q, r, idx = q[m], r[m], np.flatnonzero(m)
    az = np.degrees(np.arctan2(q[:, 1], q[:, 0]))          # -180..180
    el = np.degrees(np.arcsin(np.clip(q[:, 2] / r, -1, 1)))  # -90..90
    W = int(round(360 / deg_per_px)); H = int(round(180 / deg_per_px))
    col = np.clip(((az + 180) / deg_per_px).astype(int), 0, W - 1)
    row = np.clip(((90 - el) / deg_per_px).astype(int), 0, H - 1)
    key = row.astype(np.int64) * W + col
    order = np.lexsort((r, key)); ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    img = np.full((H * W, 3), 255, np.uint8); I = np.full(H * W, np.nan, np.float32); L = np.full(H * W, -1, np.int16)
    img[key[sel]] = rgb[idx[sel]]; I[key[sel]] = inten[idx[sel]]
    if lab is not None:
        L[key[sel]] = lab[idx[sel]]
    filled = len(sel) / (H * W)
    return img.reshape(H, W, 3), I.reshape(H, W), L.reshape(H, W), filled


def stretch(I, lo=2, hi=98):
    """Intensity -> 0..255 grey with a percentile stretch (empty pixels white)."""
    v = I[~np.isnan(I)]
    if len(v) == 0:
        return np.full(I.shape, 255, np.uint8)
    a, b = np.percentile(v, [lo, hi]); b = max(b, a + 1e-6)
    g = np.clip((I - a) / (b - a), 0, 1) * 255
    g[np.isnan(I)] = 255
    return g.astype(np.uint8)


def floor_ceiling(z, step=0.02):
    """Floor and ceiling height without labels: the most populated 2 cm height bin in the lowest 30 % of the height
    range (floor) and between 2.0 and 4.5 m above it (ceiling) - floors and ceilings are large horizontal surfaces."""
    lo, hi = float(np.percentile(z, 0.1)), float(np.percentile(z, 99.9))
    edges = lo - step + step * np.arange(int((hi - lo) / step) + 4)      # one spare bin below and two above
    h, _ = np.histogram(z, bins=edges)
    c = 0.5 * (edges[:-1] + edges[1:])
    low = c < lo + 0.3 * (hi - lo)
    floor = float(c[low][np.argmax(h[low])])
    high = (c > floor + 2.0) & (c < floor + 4.5)                    # ceiling: 2.0-4.5 m above the floor (ignores stairwells)
    if not high.any(): high = c > hi - 0.3 * (hi - lo)
    return floor, float(c[high][np.argmax(h[high])])


def scanner_position(xyz, floor_z, height, cell=0.02):
    """Scanner (x, y) from the unscanned disc on the floor under the tripod, found without labels: floor points =
    within 3 cm of the floor height. Falls back to the centre of the floor. Returns (origin, how)."""
    fm = np.abs(xyz[:, 2] - floor_z) < 0.03
    lab = np.where(fm, 12, -1)
    e = estimate_scanner_xy(xyz, lab, 12, cell)
    if e is not None and e[2] >= 0.15:
        return np.array([e[0], e[1], floor_z + height]), f"tripod disc r={e[2]:.2f} m"
    f = xyz[fm]
    return np.array([np.median(f[:, 0]), np.median(f[:, 1]), floor_z + height]), "floor centre (no tripod disc)"




def visibility(xyz, o, deg=0.4, tol_abs=0.05, tol_rel=0.03, rmin_valid=0.3):
    """Fraction of points seen from o: per (azimuth, elevation) bin only the nearest points are visible. From the
    real scanner position every scanned point is visible, from a wrong position some hide behind others."""
    q = xyz - o
    r = np.sqrt((q * q).sum(1))
    m = r > rmin_valid
    q, r = q[m], r[m]
    az = np.arctan2(q[:, 1], q[:, 0]); el = np.arcsin(np.clip(q[:, 2] / r, -1, 1))
    k = np.radians(deg)
    W = int(np.ceil(2 * np.pi / k)) + 1
    key = ((el + np.pi / 2) / k).astype(np.int64) * W + ((az + np.pi) / k).astype(np.int64)
    rmin = np.full(key.max() + 1, np.inf)
    np.minimum.at(rmin, key, r)
    return float((r <= rmin[key] + np.maximum(tol_abs, tol_rel * r)).mean())


def scan_order_test(xyz, n=200000):
    """Is the file in scanner order? Median distance between consecutive points (m)."""
    d = np.linalg.norm(np.diff(xyz[:n], axis=0), axis=1)
    return float(np.median(d))


def scanner_by_scanlines(xyz, chunk=100, min_xy=0.15, max_res=0.02, iters=3):
    """If points are stored in scan order, every vertical scan line lies in a vertical plane through the scanner:
    intersect the xy lines of many short runs of consecutive points (least squares, outliers removed)."""
    N, C = [], []
    for s in range(0, len(xyz) - chunk, chunk):
        p = xyz[s:s + chunk]
        if np.ptp(p[:, :2], axis=0).max() < min_xy: continue                       # needs radial movement (floor / ceiling)
        if np.linalg.norm(np.diff(p, axis=0), axis=1).max() > 0.5: continue        # run crosses a jump
        q = p[:, :2] - p[:, :2].mean(0)
        u, sv, vt = np.linalg.svd(q, full_matrices=False)
        if sv[1] / max(sv[0], 1e-9) > 0.05 or np.sqrt(sv[1] ** 2 / chunk) > max_res: continue
        d = vt[0]; N.append([-d[1], d[0]]); C.append(p[:, :2].mean(0))
    if len(N) < 20: return None, 0
    N, C = np.array(N), np.array(C); keep = np.ones(len(N), bool)
    for _ in range(iters):
        A = N[keep]; b = (N[keep] * C[keep]).sum(1)
        x = np.linalg.lstsq(A, b, rcond=None)[0]
        res = np.abs((N * (x - C)).sum(1)); keep = res < max(0.05, np.percentile(res[keep], 80))
    return x, dict(lines=int(keep.sum()), of=len(N), median_residual_m=float(np.median(res[keep])))


def spacing_sample(xyz, rng, n=200000):
    """Nearest-neighbour spacing of a random sample of points, measured in the FULL cloud (scanner independent)."""
    from scipy.spatial import cKDTree
    i = rng.choice(len(xyz), min(n, len(xyz)), replace=False)
    d, _ = cKDTree(xyz).query(xyz[i], k=2)
    s = d[:, 1]; ok = s > 1e-5
    return xyz[i][ok], s[ok]


def spacing_score(P, s, o):
    """A terrestrial scanner samples equal angles, so point spacing grows in proportion to the range from the real
    scanner: log(spacing) - log(range) is (nearly) constant there. Score = its robust spread (lower is better)."""
    r = np.linalg.norm(P - o, axis=1)
    v = np.log(s) - np.log(np.maximum(r, 0.05))
    med = np.median(v)
    return float(np.median(np.abs(v - med)))


def scanner_by_spacing(P, s, floor_z, height=1.6):
    """Grid search of spacing_score in x, y (height fixed: the score hardly depends on it): 0.5 m grid over the
    bounding box of P, then 0.1 m, then 0.02 m."""
    lo, hi = np.percentile(P[:, :2], 1, axis=0), np.percentile(P[:, :2], 99, axis=0)
    best = (np.inf, None)
    for x in np.arange(lo[0], hi[0], 0.5):
        for y in np.arange(lo[1], hi[1], 0.5):
            o = np.array([x, y, floor_z + height]); v = spacing_score(P, s, o)
            if v < best[0]: best = (v, o)
    for step, span in ((0.1, 0.5), (0.02, 0.1)):
        o0 = best[1]
        for dx in np.arange(-span, span + 1e-9, step):
            for dy in np.arange(-span, span + 1e-9, step):
                o = np.array([o0[0] + dx, o0[1] + dy, o0[2]]); v = spacing_score(P, s, o)
                if v < best[0]: best = (v, o)
    return best[1], best[0]

In [ ]:
# ---- self-test on a synthetic scan (known scanner position) ----
def _selftest():
    O = np.array([3.2, 4.7, 1.55]); rs = np.random.RandomState(0); pts = []
    box_lo, box_hi = np.array([6, 3, 0.0]), np.array([6.6, 3.6, 3.0])
    for az in np.radians(np.arange(0, 360, 0.6)):
        for el in np.radians(np.arange(-85, 85, 0.6)):
            d = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])
            with np.errstate(divide="ignore", invalid="ignore"):
                tr = np.where(d > 0, (np.array([10, 8, 3.0]) - O) / d, np.where(d < 0, -O / d, np.inf))
                t1, t2 = (box_lo - O) / d, (box_hi - O) / d
            t = tr.min(); tn, tf = np.nanmax(np.minimum(t1, t2)), np.nanmin(np.maximum(t1, t2))
            if 0 < tn < tf: t = min(t, tn)
            pts.append(O + t * d)
    xyz = np.array(pts) + rs.normal(0, 0.002, (len(pts), 3))
    x, info = scanner_by_scanlines(xyz); assert x is not None and np.linalg.norm(x - O[:2]) < 0.01, (x, info)
    P, s = spacing_sample(xyz, rs, 60000); o, v = scanner_by_spacing(P, s, 0.0)
    assert np.linalg.norm(o[:2] - O[:2]) < 0.2, o
    assert scanner_by_scanlines(xyz[rs.permutation(len(xyz))])[0] is None
    print(f"self-test passed: scan lines error {np.linalg.norm(x - O[:2]) * 1000:.2f} mm, spacing error {np.linalg.norm(o[:2] - O[:2]):.3f} m")
_selftest()

In [ ]:
# ---- every station: estimates, agreement, visible fraction ----
def raw_file(sub, s):
    rel = os.path.join(sub, s + ".npy")
    if os.path.isfile(os.path.join(RAW, rel)): return os.path.join(RAW, rel)
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset", allow_patterns=[sub + "/" + s + ".npy"], local_dir=RAW)
    return os.path.join(RAW, rel)

RES = {}; PANO = {}
rng = np.random.RandomState(CFG["seed"])
for s in CFG["gold"] + CFG["test"]:
    t0 = time.time(); sub = "Gold" if s in CFG["gold"] else "test_area"
    a = np.load(raw_file(sub, s))
    xyz = a[:, :3] - OFFSET; inten = a[:, 6].astype(np.float32); rgb = np.clip(a[:, 3:6], 0, 255).astype(np.uint8)
    del a
    fz, cz = floor_ceiling(xyz[:, 2]); H = CFG["scanner_height_m"]
    old, how = scanner_position(xyz, fz, H)
    fm = np.abs(xyz[:, 2] - fz) < 0.03
    centre = np.array([np.median(xyz[fm, 0]), np.median(xyz[fm, 1]), fz + H])
    disc = old if how.startswith("tripod") else None
    step = scan_order_test(xyz)
    xA, infoA = scanner_by_scanlines(xyz) if step < 0.1 else (None, "file not in scanner order")
    P, sp = spacing_sample(xyz, rng, CFG["spacing_n"])
    oB, scB = scanner_by_spacing(P, sp, fz, H)
    oA = None if xA is None else np.array([xA[0], xA[1], fz + H])
    sub_v = xyz[rng.choice(len(xyz), min(CFG["vis_n"], len(xyz)), replace=False)]
    cand = {"floor_centre": centre, "tripod_disc": disc, "A_scanlines": oA, "B_spacing": oB}
    R = dict(points=len(xyz), floor=round(fz, 3), ceiling=round(cz, 3), old_how=how, median_step_m=round(step, 5),
             scanlines_info=infoA, spacing_score=round(scB, 4),
             spacing_score_at_old=round(spacing_score(P, sp, old), 4),
             est={k: None if v is None else [round(float(c), 3) for c in v] for k, v in cand.items()},
             visible={k: None if v is None else round(visibility(sub_v, v), 5) for k, v in cand.items()})
    names = [k for k, v in cand.items() if v is not None]
    R["dist_xy_m"] = {f"{p}-{q}": round(float(np.linalg.norm(cand[p][:2] - cand[q][:2])), 3) for i, p in enumerate(names) for q in names[i + 1:]}
    # choice: A if it exists and agrees with B within 0.3 m (or the disc), else B; the old one stays in the json too
    if oA is not None and (np.linalg.norm(oA[:2] - oB[:2]) < 0.3 or (disc is not None and np.linalg.norm(oA[:2] - disc[:2]) < 0.3)):
        new, nhow = oA, "A_scanlines"
    else:
        new, nhow = oB, "B_spacing"
    R["new"] = [round(float(c), 3) for c in new]; R["new_how"] = nhow; R["old"] = [round(float(c), 3) for c in old]
    R["moved_xy_m"] = round(float(np.linalg.norm(new[:2] - old[:2])), 3); R["seconds"] = round(time.time() - t0, 1)
    RES[s] = R
    print(f"{s}: {how:<32} step {step*1000:7.2f} mm | A {infoA if xA is None else (np.round(xA, 2).tolist(), infoA)} | "
          f"B {np.round(oB[:2], 2).tolist()} (score {scB:.4f}, at old {R['spacing_score_at_old']}) | new = {nhow}, moved {R['moved_xy_m']} m | "
          f"distances {R['dist_xy_m']} | visible {R['visible']} | {time.time() - t0:.0f} s")
    # panoramas from the old and the new position (intensity)
    if s in CFG["test"] or s in ("S164", "S165"):
        fig, ax = plt.subplots(2, 1, figsize=(18, 9.5))
        for k, (o, ttl) in enumerate(((old, f"old: {how}"), (new, f"new: {nhow}"))):
            img, I, L, filled = render_panorama(xyz, rgb, inten, None, o, deg_per_px=CFG["pano_deg_per_px"])
            ax[k].imshow(stretch(I), cmap="gray", vmin=0, vmax=255); ax[k].set_title(f"{s} — {ttl} — filled {filled:.2f}"); ax[k].axis("off")
        plt.tight_layout(); fn = os.path.join(OUT, f"pano_{s}.png"); plt.savefig(fn, dpi=70); plt.close(fig)
    del xyz, inten, rgb, P, sp, sub_v

with open(os.path.join(OUT, "scanner_positions.json"), "w") as fh:
    json.dump({s: dict(new=R["new"], new_how=R["new_how"], old=R["old"], old_how=R["old_how"], floor=R["floor"], ceiling=R["ceiling"]) for s, R in RES.items()}, fh, indent=1)
with open(os.path.join(OUT, "summary.json"), "w") as fh:
    json.dump(dict(cfg=CFG, stations=RES), fh, indent=1)

In [ ]:
# ---- summary table, and how well A / B agree with the tripod disc where it exists ----
print(f"{'st':<5} {'old':<32} {'step mm':>8} {'new':<12} {'moved':>6} {'B-disc':>7} {'A-disc':>7} {'A-B':>6} | visible old -> new")
for s, R in RES.items():
    d = R["dist_xy_m"]
    vo = R["visible"]["tripod_disc"] if R["old_how"].startswith("tripod") else R["visible"]["floor_centre"]
    print(f"{s:<5} {R['old_how']:<32} {R['median_step_m']*1000:>8.2f} {R['new_how']:<12} {R['moved_xy_m']:>6} "
          f"{d.get('tripod_disc-B_spacing', '-'):>7} {d.get('tripod_disc-A_scanlines', '-'):>7} {d.get('A_scanlines-B_spacing', '-'):>6} | "
          f"{vo} -> {R['visible'][R['new_how']]}")
print("\nsaved:", sorted(os.listdir(OUT)))

## After the run

Record as **FEAS-005** in `docs/experiment_log.md`: for the stations with a tripod disc, the distance of A and B from the disc (this is the check of the method); for the others, how far the scanner moved and whether the panoramas from the new position look like a real scan (sharp, no smeared walls). If the new positions are right, notebook 10 version 4 uses `scanner_positions.json` (attach this notebook's output as a dataset) and the hybrid is re-evaluated on validation before any new test submission.